## Ajustante dataset FakeTrueBR

## Concatenadno datasets

In [1]:
import pandas as pd

In [ ]:
import os
import pickle
from pathlib import Path
import pandas as pd

# Se os arquivos estiverem em outra pasta, ajuste esta base:
BASE_DIR = Path(".")

FILES = [
    "COVID19.BR_raw.parquet",
    "Fake.br_raw.parquet",
    "FakeNewsSet.parquet",
    "FakeRecogna.xlsx",
    "MuMiN-PT_raw.parquet",
    "central_de_fatos.tsv",
    "fcn.plk",
    "faketruebr.csv",
    "FACTCKBR.tsv",
    "fakepedia-copus-v1.csv"
]

def print_columns(df: pd.DataFrame, label: str):
    cols = list(df.columns)
    print(f"[OK] {label} -> {len(cols)} colunas")
    print("     ", cols)

def read_any_parquet(path: Path) -> pd.DataFrame:
    # pandas tenta automaticamente o engine disponível (pyarrow/fastparquet)
    return pd.read_parquet(path)

def read_any_excel(path: Path):
    # Lê todas as sheets e retorna dict {sheet_name: df}
    xls = pd.ExcelFile(path)
    res = {}
    for sheet in xls.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        res[sheet] = df
    return res

def read_any_tsv(path: Path) -> pd.DataFrame:
    # Se precisar, ajuste o encoding
    return pd.read_csv(path, sep="\t", encoding="utf-8", engine="python", on_bad_lines="skip")

def read_any_pickle(path: Path):
    # 1) Tenta pandas.read_pickle (quando é DataFrame/Series)
    try:
        obj = pd.read_pickle(path)
        return obj
    except Exception:
        pass
    # 2) Tenta pickle bruto (pode ser dict/list/DataFrame etc.)
    with open(path, "rb") as f:
        return pickle.load(f)

def process_file(fname: str):
    path = (BASE_DIR / fname).resolve()
    if not path.exists():
        print(f"[ERRO] Arquivo não encontrado: {path}")
        return

    ext = path.suffix.lower()
    print(f"\n=== {path.name} ===")

    try:
        if ext == ".parquet":
            df = read_any_parquet(path)
            print_columns(df, path.name)

        elif ext in (".xlsx", ".xls"):
            sheets = read_any_excel(path)
            for sheet, df in sheets.items():
                print_columns(df, f"{path.name} :: aba '{sheet}'")

        elif ext in (".tsv",):
            df = read_any_tsv(path)
            print_columns(df, path.name)

        elif ext in (".pkl", ".plk"):
            obj = read_any_pickle(path)
            # Caso seja DataFrame:
            if isinstance(obj, pd.DataFrame):
                print_columns(obj, f"{path.name} (pickle->DataFrame)")
            # Caso seja Series:
            elif isinstance(obj, pd.Series):
                print(f"[OK] {path.name} (pickle->Series) -> index name: {obj.name}, length: {len(obj)}")
            # Caso seja dict de DataFrames/similares:
            elif isinstance(obj, dict):
                print(f"[OK] {path.name} (pickle->dict) -> chaves: {list(obj.keys())}")
                # Se houver DataFrames dentro, mostre colunas:
                for k, v in obj.items():
                    if isinstance(v, pd.DataFrame):
                        print_columns(v, f"{path.name} :: chave '{k}'")
                    else:
                        vtype = type(v).__name__
                        print(f"     chave '{k}' tipo {vtype}")
            else:
                # Outro tipo
                print(f"[OK] {path.name} (pickle->{type(obj).__name__}) -> conteúdo não-DataFrame")

        else:
            # Fallback: tenta CSV padrão
            try:
                df = pd.read_csv(path)
                print_columns(df, f"{path.name} (CSV fallback)")
            except Exception as e_csv:
                print(f"[ERRO] Formato não reconhecido para {path.name} e CSV fallback falhou: {e_csv}")

    except Exception as e:
        print(f"[ERRO] Falha ao ler {path.name}: {e}")

if __name__ == "__main__":
    for f in FILES:
        process_file(f)

In [ ]:
import os
from pathlib import Path
import pickle
import pandas as pd
import numpy as np

# ====== caminhos (ajuste se necessário) ======
BASE = Path(".")
PATHS = {
    "COVID19BR"   : BASE / "COVID19.BR_raw.parquet",
    "FAKEBR"      : BASE / "Fake.br_raw.parquet",
    "FNEWSSET"    : BASE / "FakeNewsSet.parquet",
    "FRECOGNA"    : BASE / "FakeRecogna.xlsx",  # leitura robusta adicionada
    "MUMINPT"     : BASE / "MuMiN-PT_raw.parquet",
    "CENTRALFATOS": BASE / "central_de_fatos.tsv",
    "FCN"         : BASE / "fcn.plk",
    "FAKETRUEBR":  BASE / "faketruebr.csv",
    "FACTCKBR":  BASE / "FACTCKBR.tsv",
    "BOATOSBR":  BASE / "boatos_br_corpus.json"
}

OUT_PARQUET = BASE / "Unified_PTBR_FakeNews.parquet"
OUT_CSV     = BASE / "Unified_PTBR_FakeNews.csv"

# ====== normalização ======
def to_FAKE_REAL(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    # numéricos comuns
    if s in {"1","1.0","+1"}: return "FAKE"
    if s in {"0","0.0"}:      return "REAL"
    if s in {"-1","-1.0"}:    return "REAL"
    # strings pt/en
    if s in {"fake","falso","falsa"}: return "FAKE"
    if s in {"real","verdadeiro","verdadeira","verdade", "true"}: return "REAL"
    # fallback literal
    if s.upper() in {"FAKE","REAL"}: return s.upper()
    return np.nan

def norm_text(x):
    if pd.isna(x): return ""
    return " ".join(str(x).split())

def make_idx(prefix, id_value, pad=6):
    if id_value is None or (isinstance(id_value, float) and np.isnan(id_value)): return None
    sid = str(id_value).strip().replace(" ", "_").replace("/", "-")
    if sid.isdigit(): sid = sid.zfill(pad)
    return f"{prefix}_{sid}"

frames = []

# ====== 1) COVID19.BR_raw.parquet ======
p = PATHS["COVID19BR"]
if p.exists():
    df = pd.read_parquet(p)
    sub = pd.DataFrame({
        "idx": [make_idx("COVID19BR", i) for i in range(len(df))],
        "dataset_id": "COVID19BR",
        "text_no_url": df["text_no_url"].map(norm_text),
        "label": df["label"].map(to_FAKE_REAL),
    })
    frames.append(sub)
else:
    print("[WARN] não encontrei:", p)

# ====== 2) Fake.br_raw.parquet ======
p = PATHS["FAKEBR"]
if p.exists():
    df = pd.read_parquet(p)
    sub = pd.DataFrame({
        "idx": [make_idx("FAKEBR", i) for i in range(len(df))],
        "dataset_id": "FAKEBR",
        "text_no_url": df["text_no_url"].map(norm_text),
        "label": df["label"].map(to_FAKE_REAL),
    })
    frames.append(sub)
else:
    print("[WARN] não encontrei:", p)

# ====== 3) FakeNewsSet.parquet ======
# text_no_url <- title ; label 0->REAL, 1->FAKE
p = PATHS["FNEWSSET"]
if p.exists():
    df = pd.read_parquet(p)
    lab = df["label"].map(lambda v: "REAL" if str(v).strip() in {"0","0.0"} else ("FAKE" if str(v).strip() in {"1","1.0"} else np.nan))
    ids = (df["id"].astype(str) if "id" in df.columns else pd.Series(range(len(df))).astype(str))
    sub = pd.DataFrame({
        "idx": [make_idx("FNEWSSET", v) for v in ids],
        "dataset_id": "FNEWSSET",
        "text_no_url": df["title"].map(norm_text),
        "label": lab,
    })
    frames.append(sub)
else:
    print("[WARN] não encontrei:", p)

# ====== 4) FakeRecogna.xlsx ====== (robusto)
# text_no_url <- Noticia ; label <- Classe (0->REAL, 1->FAKE)
p = PATHS["FRECOGNA"]
def map_classe_to_FR(v):
    if pd.isna(v):
        return np.nan
    # tenta via float (lida com 0.0 / 1.0)
    try:
        f = float(v)
        if np.isclose(f, 0.0):
            return "REAL"
        if np.isclose(f, 1.0):
            return "FAKE"
    except Exception:
        pass
    # fallback string
    s = str(v).strip().lower()
    if s in {"0", "0.0"}:
        return "REAL"
    if s in {"1", "1.0"}:
        return "FAKE"
    return np.nan
if p.exists():
    tmp = pd.read_excel(p)

    # normaliza cabeçalhos com .strip()
    cols = {c.strip(): c for c in tmp.columns.astype(str)}

    if ("Noticia" in cols or "Notícia" in cols) and ("Classe" in cols):
        noticia_col = cols.get("Noticia", cols.get("Notícia"))
        lab = tmp[cols["Classe"]].apply(map_classe_to_FR)
        sub = pd.DataFrame({
            "idx": [make_idx("FRECOGNA", i) for i in range(len(tmp))],
            "dataset_id": "FRECOGNA",
            "text_no_url": tmp[noticia_col].map(norm_text),
            "label": lab,
        })
        frames.append(sub)
    else:
        print(f"[WARN] '{p.name}' primeira sheet sem colunas esperadas (Noticia/Classe). Encontradas: {list(tmp.columns)}")
else:
    print("[WARN] não encontrei:", p)


# ====== 5) MuMiN-PT_raw.parquet ======
p = PATHS["MUMINPT"]
if p.exists():
    df = pd.read_parquet(p)
    sub = pd.DataFrame({
        "idx": [make_idx("MUMINPT", i) for i in range(len(df))],
        "dataset_id": "MUMINPT",
        "text_no_url": df["text_no_url"].map(norm_text),
        "label": df["label"].map(to_FAKE_REAL),
    })
    frames.append(sub)
else:
    print("[WARN] não encontrei:", p)

# ====== 6) central_de_fatos.tsv ======
# text_no_url <- review_text ; is_fake 0->REAL, 1->FAKE
p = PATHS["CENTRALFATOS"]
if p.exists():
    df = pd.read_csv(p, sep="\t", encoding="utf-8", engine="python", on_bad_lines="skip")
    lab = df["is_fake"].map(lambda v: "REAL" if str(v).strip()=="0" else ("FAKE" if str(v).strip()=="1" else np.nan))
    ids = (df["review_id"].astype(str) if "review_id" in df.columns else pd.Series(range(len(df))).astype(str))
    sub = pd.DataFrame({
        "idx": [make_idx("CENTRALFATOS", v) for v in ids],
        "dataset_id": "CENTRALFATOS",
        "text_no_url": df["review_text"].map(norm_text),
        "label": lab,
    })
    frames.append(sub)
else:
    print("[WARN] não encontrei:", p)

# ====== 7) fcn.plk ======
# text_no_url <- text ; class 1->FAKE, -1->REAL
p = PATHS["FCN"]
if p.exists():
    try:
        obj = pd.read_pickle(p)
    except Exception:
        with open(p, "rb") as f:
            obj = pickle.load(f)

    if isinstance(obj, pd.DataFrame):
        df = obj
    elif isinstance(obj, dict) and "data" in obj and isinstance(obj["data"], pd.DataFrame):
        df = obj["data"]
    else:
        raise ValueError("Objeto no pickle não é DataFrame e não tem chave 'data' com DataFrame.")

    def map_fcn_class(v):
        s = str(v).strip()
        if s in {"1","1.0","+1"}:  return "FAKE"
        if s in {"-1","-1.0"}:     return "REAL"
        return np.nan

    sub = pd.DataFrame({
        "idx": [make_idx("FCN", i) for i in range(len(df))],
        "dataset_id": "FCN",
        "text_no_url": df["text"].map(norm_text),
        "label": df["class"].map(map_fcn_class),
    })
    frames.append(sub)
else:
    print("[WARN] não encontrei:", p)

# ====== 8) 300-noticias-v2-filtradas.csv ====== (NOVO)
# text_no_url <- Texto_Original ; label <- Rótulo (valores 'real'/'falso' a normalizar)
p = PATHS["TRE300"]
if p.exists():
    df = pd.read_csv(p, encoding="utf-8", engine="python")
    cols = {c.strip(): c for c in df.columns.astype(str)}
    if ("Texto_Original" in cols or "text" in cols) and ("Rótulo" in cols or "Rotulo" in cols or "rótulo" in cols):
        text_col = cols.get("Texto_Original", cols.get("text"))
        rotulo_col = cols.get("Rótulo", cols.get("Rotulo", cols.get("rótulo")))
        # idx: usar ID se existir, senão sequencial
        if "ID" in cols:
            ids = df[cols["ID"]].astype(str)
        else:
            ids = pd.Series(range(len(df))).astype(str)
        lab = df[rotulo_col].map(lambda v: "REAL" if str(v).strip().lower() in {"real","verdadeiro","verdadeira"} else
                                           ("FAKE" if str(v).strip().lower() in {"falso","falsa","fake"} else np.nan))
        sub = pd.DataFrame({
            "idx": [make_idx("TRE300", v) for v in ids],
            "dataset_id": "TRE300",
            "text_no_url": df[text_col].map(norm_text),
            "label": lab,
        })
        frames.append(sub)
    else:
        print(f"[WARN] {p.name} sem colunas esperadas (Texto_Original / Rótulo).")
else:
    print("[WARN] não encontrei:", p)

# ====== 9) FAKETRUEBR.csv ======
p = PATHS["FAKETRUEBR"]
if p.exists():
    # Carrega o CSV bruto
    df_final = pd.read_csv(p)
    print(df_final.columns)

    # --- Adapta para o formato padrão do pipeline ---
    sub = pd.DataFrame({
        "idx": [make_idx("FAKETRUEBR", i) for i in range(len(df_final))],
        "dataset_id": "FAKETRUEBR",
        "text_no_url": df_final["text"].map(norm_text),
        "label": df_final["label"].map(to_FAKE_REAL),
    })

    frames.append(sub)
else:
    print("[WARN] não encontrei:", p)

# ====== 10) FAKETRUEBR.csv ======
p = PATHS["FACTCKBR"]
if p.exists():
    # Carrega o TSV bruto (Adicionado 'p' e o separador de tabulação)
    df_final = pd.read_csv(p, sep='\t')
    print(df_final.columns)

    # --- Adapta para o formato padrão do pipeline ---
    sub = pd.DataFrame({
        "idx": [make_idx("FACTCKBR", i) for i in range(len(df_final))],
        "dataset_id": "FACTCKBR",
        # Concatena title + \n + reviewBody
        "text_no_url": df_final['title'] + "\n" + df_final['reviewBody'],
        "label": df_final["alternativeName"].map(to_FAKE_REAL),
    })

    frames.append(sub)
else:
    print("[WARN] não encontrei:", p)
    

# ====== 10) FAKETRUEBR.csv ======
p = PATHS["FACTCKBR"]
if p.exists():
    # Carrega o TSV bruto (Adicionado 'p' e o separador de tabulação)
    df_final = pd.read_csv(p, sep='\t')
    print(df_final.columns)

    # --- Adapta para o formato padrão do pipeline ---
    sub = pd.DataFrame({
        "idx": [make_idx("FACTCKBR", i) for i in range(len(df_final))],
        "dataset_id": "FACTCKBR",
        # Concatena title + \n + reviewBody
        "text_no_url": df_final['title'] + "\n" + df_final['reviewBody'],
        "label": df_final["alternativeName"].map(to_FAKE_REAL),
    })

    frames.append(sub)
else:
    print("[WARN] não encontrei:", p)

# ====== 11) BOATOS.json ======
p = PATHS["BOATOSBR"]
if p.exists():
    # Carrega o TSV bruto (Adicionado 'p' e o separador de tabulação)
    df_final = pd.read_json(p)
    print(df_final.columns)

    # --- Adapta para o formato padrão do pipeline ---
    sub = pd.DataFrame({
        "idx": [make_idx("BOATOSBR", i) for i in range(len(df_final))],
        "dataset_id": "BOATOSBR",
        # Concatena title + \n + reviewBody
        "text_no_url": df_final['texto'],
        "label": df_final["rotulo"].map(to_FAKE_REAL),
    })

    frames.append(sub)
else:
    print("[WARN] não encontrei:", p)

# ====== concatena e saneia ======
if not frames:
    raise RuntimeError("Nenhum dataset foi carregado. Verifique os caminhos.")

unified = pd.concat(frames, ignore_index=True)

# remove linhas sem texto/label e resolve duplicatas de idx
unified["text_no_url"] = unified["text_no_url"].fillna("").astype(str).str.strip()
valid = (unified["text_no_url"].str.len() > 0) & (unified["label"].isin(["FAKE","REAL"]))
unified = unified.loc[valid].copy()

if unified["idx"].duplicated().any():
    unified["dup_count"] = unified.groupby("idx").cumcount()
    mask = unified["dup_count"] > 0
    unified.loc[mask, "idx"] = unified.loc[mask, "idx"] + "_" + unified.loc[mask, "dup_count"].astype(str)
    unified.drop(columns=["dup_count"], inplace=True)

unified = unified[["idx","dataset_id","text_no_url","label"]].reset_index(drop=True)

# ====== salva ======
unified.to_parquet(OUT_PARQUET, index=False)
unified.to_csv(OUT_CSV, index=False, encoding="utf-8")
print("✅ salvo:")
print(" -", OUT_PARQUET)
print(" -", OUT_CSV)

# ====== resumo ======
print("\nResumo:")
print("linhas:", len(unified))
print("por dataset_id:", unified["dataset_id"].value_counts().to_dict())
print("labels:", unified["label"].value_counts().to_dict())
print("exemplo:")
print(unified.head(5))
